In [4]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 04 — NLP e Análise de Sentimento
# ============================================================
# Objetivo: Analisar o sentimento dos reviews usando léxico
# manual calibrado para o domínio cosmético PT-BR.
#
# Entregas:
# - sentimento_score  → float entre -1 e 1
# - classificacao     → Positivo / Neutro / Negativo
# - palavras-chave    → top palavras por faixa de rating
# ============================================================

import pandas as pd
import re
import nltk
from collections import Counter
from pathlib import Path

# Caminhos do projeto
ROOT = Path().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed"

# Carrega o dataset enriquecido do Notebook 03
df = pd.read_csv(PROCESSED_PATH / "reviews_enriquecidos.csv", encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"])

print(f"✅ Dataset carregado: {df.shape[0]:,} registros")
print(f"📋 Colunas: {list(df.columns)}")

✅ Dataset carregado: 202,785 registros
📋 Colunas: ['date', 'rating', 'content', 'product_url', 'qtd_palavras', 'review_curto', 'content_original', 'produto_id', 'slug', 'nome_produto', 'categoria']


In [6]:
# ============================================================
# Download dos recursos necessários do NLTK
# ============================================================

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Stopwords em português
STOPWORDS_PT = set(stopwords.words("portuguese"))

# Stopwords adicionais específicas do domínio
STOPWORDS_EXTRA = {
    "ml", "kg", "gr", "g", "l", "un", "und", "kit",
    "produto", "produtos", "item", "itens", "compra",
    "comprei", "recebi", "chegou", "entrega", "vendedor",
    "mercado", "livre", "prazo", "embalagem", "original"
}

STOPWORDS_PT.update(STOPWORDS_EXTRA)

print(f"✅ NLTK configurado")
print(f"📚 Total de stopwords: {len(STOPWORDS_PT)}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\edigu\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\edigu\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\edigu\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


✅ NLTK configurado
📚 Total de stopwords: 230


In [8]:
# ============================================================
# Tokenização e remoção de stopwords
# Prepara o texto para o cálculo de sentimento
# ============================================================

def preprocessar_texto(texto):
    """
    Tokeniza o texto e remove stopwords.
    Retorna lista de tokens limpos.
    """
    if pd.isna(texto) or str(texto).strip() == "":
        return []

    # Tokeniza
    tokens = word_tokenize(str(texto).lower(), language="portuguese")

    # Remove stopwords e tokens não alfabéticos
    tokens_limpos = [
        t for t in tokens
        if t.isalpha() and t not in STOPWORDS_PT and len(t) > 2
    ]

    return tokens_limpos

# Aplica no dataset
df["tokens"] = df["content"].apply(preprocessar_texto)

# Diagnóstico
media_tokens = df["tokens"].apply(len).mean()
zero_tokens  = (df["tokens"].apply(len) == 0).sum()

print(f"✅ Tokenização concluída")
print(f"📊 Média de tokens por review: {media_tokens:.1f}")
print(f"⚠️  Reviews sem tokens após limpeza: {zero_tokens:,}")
print(f"\nExemplo de tokenização:")

for i, row in df[["content", "tokens"]].head(5).iterrows():
    print(f"\n  Original : {row['content']}")
    print(f"  Tokens   : {row['tokens']}")

✅ Tokenização concluída
📊 Média de tokens por review: 5.3
⚠️  Reviews sem tokens após limpeza: 1,163

Exemplo de tokenização:

  Original : top.
  Tokens   : ['top']

  Original : produto bom, cumpre o que promete.
  Tokens   : ['bom', 'cumpre', 'promete']

  Original : ótima qualidade.
  Tokens   : ['ótima', 'qualidade']

  Original : bom.
  Tokens   : ['bom']

  Original : atendeu minhas expectativas.
  Tokens   : ['atendeu', 'expectativas']


In [15]:
# mostrar as colunas content e tokens lado a lado
df[["content", "tokens"]].tail(10)

,content,tokens
202775,"para meu cabelo não deu certo ,pq já é seco fi...","[cabelo, deu, certo, seco, ficou, ainda]"
202776,não resolve nada para oleosidade.,"[resolve, nada, oleosidade]"
202777,excelente produto.,[excelente]
202778,produto excelente.,[excelente]
202779,super bom.,"[super, bom]"
202780,não é muito cheiroso e não deixa o cabelo maci...,"[cheiroso, deixa, cabelo, macio, espero, menos..."
202781,muito bom ! recomendo.,"[bom, recomendo]"
202782,maravilhoso! comprarei mais vezes.,"[maravilhoso, comprarei, vezes]"
202783,não é original!. comprei um menor na loja de c...,"[menor, loja, cosméticos, antes, comprar, obti..."
202784,"é bom, mais já usei produtos bem melhores e ma...","[bom, usei, bem, melhores, baratos]"


In [12]:
# ============================================================
# Léxico manual calibrado para reviews cosméticos PT-BR
# Pesos: +2 (forte positivo), +1 (moderado positivo)
#        -2 (forte negativo), -1 (moderado negativo)
# ============================================================

LEXICO_POSITIVO = {
    # Peso +2 — fortemente positivos
    "maravilhoso": 2, "maravilhosa": 2, "excelente": 2,
    "incrivel": 2, "perfeito": 2, "perfeita": 2,
    "amei": 2, "adorei": 2, "otimo": 2, "otima": 2,
    "fantastico": 2, "fantastica": 2, "magico": 2, "magica": 2,
    "surpreendente": 2, "espetacular": 2, "sensacional": 2,
    "melhor": 2, "aprovado": 2, "aprovada": 2,
    "top": 2, "recomendo": 2, "recomendado": 2,

    # Peso +1 — moderadamente positivos
    "bom": 1, "boa": 1, "gostei": 1, "gosto": 1,
    "bonito": 1, "bonita": 1, "cheiroso": 1, "cheirosa": 1,
    "hidratado": 1, "hidratada": 1, "hidrata": 1,
    "macio": 1, "macia": 1, "brilhoso": 1, "brilhante": 1,
    "suave": 1, "leve": 1, "funciona": 1, "funcionou": 1,
    "cumpre": 1, "atendeu": 1, "valeu": 1, "vale": 1,
    "satisfeito": 1, "satisfeita": 1, "feliz": 1,
    "contente": 1, "rapido": 1, "pratico": 1, "pratica": 1,
    "lindo": 1, "linda": 1, "agradavel": 1, "eficiente": 1,
    "eficaz": 1, "nutritivo": 1, "nutritiva": 1,
    "restaurou": 1, "recuperou": 1, "fortaleceu": 1,
}

LEXICO_NEGATIVO = {
    # Peso -2 — fortemente negativos
    "horrivel": -2, "pessimo": -2, "pessima": -2,
    "terrivel": -2, "odiei": -2, "detestei": -2,
    "lixo": -2, "vergonha": -2, "fraude": -2,
    "enganoso": -2, "enganosa": -2, "decepcionante": -2,
    "estragou": -2, "queimou": -2, "arrependi": -2,
    "ressecou": -2, "quebrou": -2,

    # Peso -1 — moderadamente negativos
    "ruim": -1, "fraco": -1, "fraca": -1,
    "problema": -1, "problemas": -1, "demorou": -1,
    "atrasou": -1, "danificou": -1, "irritou": -1,
    "cocou": -1, "oleoso": -1, "oleosa": -1,
    "pesado": -1, "pesada": -1, "caro": -1,
    "decepcionei": -1, "decepcionou": -1, "fraquinho": -1,
    "fraquinha": -1, "regular": -1, "mediano": -1,
    "mediana": -1, "cheiro-ruim": -1, "resseca": -1,
    "caindo": -1, "queda": -1, "nao-gostei": -1,
}

# Negadores — invertem o sinal da próxima palavra
NEGADORES = {"nao", "nunca", "jamais", "nem", "tampouco", "sequer"}

print(f"✅ Léxico carregado")
print(f"📗 Palavras positivas: {len(LEXICO_POSITIVO)}")
print(f"📕 Palavras negativas: {len(LEXICO_NEGATIVO)}")
print(f"🔄 Negadores:         {len(NEGADORES)}")

✅ Léxico carregado
📗 Palavras positivas: 63
📕 Palavras negativas: 44
🔄 Negadores:         6


In [18]:
# ============================================================
# Calcula o score de sentimento para cada review
# Leva em conta negadores (ex: "não gostei" → negativo)
# Score normalizado entre -1 e 1
# ============================================================

def calcular_score(tokens):
    """
    Percorre os tokens do review.
    Aplica pesos do léxico e inverte sinal após negadores.
    Normaliza o score pelo número de tokens com peso.
    Retorna float entre -1 e 1.
    """
    if not tokens:
        return 0.0

    score_bruto   = 0
    total_pesos   = 0
    proximo_negado = False

    for token in tokens:
        # Verifica se é negador
        if token in NEGADORES:
            proximo_negado = True
            continue

        # Verifica no léxico positivo
        if token in LEXICO_POSITIVO:
            peso = LEXICO_POSITIVO[token]
            score_bruto += (-peso if proximo_negado else peso)
            total_pesos += abs(peso)
            proximo_negado = False

        # Verifica no léxico negativo
        elif token in LEXICO_NEGATIVO:
            peso = LEXICO_NEGATIVO[token]
            score_bruto += (-peso if proximo_negado else peso)
            total_pesos += abs(peso)
            proximo_negado = False

        else:
            proximo_negado = False

    # Normaliza entre -1 e 1
    if total_pesos == 0:
        return 0.0

    score_normalizado = score_bruto / total_pesos
    # Garante que fica entre -1 e 1
    return round(max(-1.0, min(1.0, score_normalizado)), 4)

# Aplica no dataset
df["sentimento_score"] = df["tokens"].apply(calcular_score)

print(f"✅ Scores calculados")
print(f"\n📊 Distribuição dos scores:")
print(df["sentimento_score"].describe().round(4))
print(f"\nExemplos:")
display(df[["content", "sentimento_score"]].tail(5))

✅ Scores calculados

📊 Distribuição dos scores:
count    202785.0000
mean          0.7047
std           0.4916
min          -1.0000
25%           0.0000
50%           1.0000
75%           1.0000
max           1.0000
Name: sentimento_score, dtype: float64

Exemplos:


,content,sentimento_score
202780,não é muito cheiroso e não deixa o cabelo maci...,0.3333
202781,muito bom ! recomendo.,1.0000
202782,maravilhoso! comprarei mais vezes.,1.0000
202783,não é original!. comprei um menor na loja de c...,-1.0000
202784,"é bom, mais já usei produtos bem melhores e ma...",1.0000


In [19]:
# ============================================================
# Converte o score numérico em classificação textual
#
# Positivo  → score >  0.05
# Neutro    → score entre -0.05 e 0.05
# Negativo  → score < -0.05
#
# O threshold de 0.05 evita que reviews sem palavras
# do léxico sejam classificados incorretamente
# ============================================================

def classificar_sentimento(score):
    if score > 0.05:
        return "Positivo"
    elif score < -0.05:
        return "Negativo"
    else:
        return "Neutro"

df["classificacao_sentimento"] = df["sentimento_score"].apply(classificar_sentimento)

# Diagnóstico
print("=" * 55)
print("DISTRIBUIÇÃO DE SENTIMENTOS")
print("=" * 55)

dist_sent = df["classificacao_sentimento"].value_counts()
pct_sent  = (dist_sent / len(df) * 100).round(2)

diagnostico_sent = pd.DataFrame({
    "Quantidade": dist_sent,
    "% do Total": pct_sent
})
print(diagnostico_sent)

print("\n" + "=" * 55)
print("COERÊNCIA: SENTIMENTO x RATING")
print("=" * 55)

# Verifica se ratings altos têm sentimento positivo
coerencia = df.groupby("rating")["sentimento_score"].mean().round(4)
print(coerencia)
print("\n💡 Esperado: score cresce conforme rating aumenta")

DISTRIBUIÇÃO DE SENTIMENTOS
                          Quantidade  % do Total
classificacao_sentimento                        
Positivo                      148755       73.36
Neutro                         49879       24.60
Negativo                        4151        2.05

COERÊNCIA: SENTIMENTO x RATING
rating
1    0.0818
2    0.2482
3    0.3413
4    0.6168
5    0.7559
Name: sentimento_score, dtype: float64

💡 Esperado: score cresce conforme rating aumenta


In [26]:
# mostrar o content e score lado a lado
df[["content", "rating", "sentimento_score", "classificacao_sentimento"]].tail(30)

,content,rating,sentimento_score,classificacao_sentimento
202755,produto excelente.,5,1.0000,Positivo
202756,"muito bom, efetivo no que promete, além de ref...",5,1.0000,Positivo
202757,muito bom.,5,1.0000,Positivo
202758,eu achei o produto excelente. meu esposo tem m...,5,0.3333,Positivo
202759,limpo a cuca!.,5,0.0000,Neutro
202760,gostei!!!.,5,1.0000,Positivo
202761,é maravilhoso!!! acaba com a caspa já na prime...,5,1.0000,Positivo
202762,melhor shampoo para caspa e oleosidade. refres...,5,1.0000,Positivo
202763,shampoo maravilhoso.,5,1.0000,Positivo
202764,super recomendo.,5,1.0000,Positivo


In [27]:
# ============================================================
# Top palavras em reviews de 1⭐ vs 5⭐
# Um dos insights mais visuais do projeto no Power BI
# ============================================================

def top_palavras(dataframe, rating, n=20):
    """
    Retorna as N palavras mais frequentes
    para uma faixa de rating específica.
    """
    tokens_rating = dataframe[
        dataframe["rating"] == rating
    ]["tokens"].explode()

    contagem = Counter(tokens_rating.dropna())
    return pd.DataFrame(
        contagem.most_common(n),
        columns=["palavra", "frequencia"]
    )

# Gera para cada rating
for estrelas in [1, 2, 3, 4, 5]:
    top = top_palavras(df, estrelas, n=15)
    print(f"\n{'=' * 40}")
    print(f"⭐ TOP 15 PALAVRAS — RATING {estrelas}")
    print(f"{'=' * 40}")
    print(top.to_string(index=False))


⭐ TOP 15 PALAVRAS — RATING 1
 palavra  frequencia
  cabelo        2488
    veio        1234
  cheiro         999
  gostei         931
    nada         673
    ruim         474
  parece         456
   ficou         434
horrível         411
     bom         399
    usei         382
   achei         366
 shampoo         364
     pra         364
     bem         346

⭐ TOP 15 PALAVRAS — RATING 2
  palavra  frequencia
   cabelo        1103
   gostei         492
   cheiro         366
     veio         348
      bom         332
    achei         323
     nada         219
    ficou         209
      bem         205
      pra         183
resultado         176
     fica         168
   parece         164
    deixa         157
     usei         143

⭐ TOP 15 PALAVRAS — RATING 3
  palavra  frequencia
   cabelo        2061
      bom        1204
    achei         747
   cheiro         713
   gostei         680
      bem         555
     veio         482
      pra         421
    deixa         353
  

In [28]:
# ============================================================
# Validação completa antes da exportação
# ============================================================

print("=" * 55)
print("RELATÓRIO DE VALIDAÇÃO — NLP")
print("=" * 55)

print(f"\n📊 SCORES")
print(f"  Score médio geral:     {df['sentimento_score'].mean():.4f}")
print(f"  Score reviews 1⭐:     {df[df['rating']==1]['sentimento_score'].mean():.4f}")
print(f"  Score reviews 5⭐:     {df[df['rating']==5]['sentimento_score'].mean():.4f}")

print(f"\n🏷️  CLASSIFICAÇÕES")
for classe in ["Positivo", "Neutro", "Negativo"]:
    qtd = (df["classificacao_sentimento"] == classe).sum()
    pct = qtd / len(df) * 100
    print(f"  {classe:10}: {qtd:,} ({pct:.2f}%)")

print(f"\n🔍 CASOS EXTREMOS")
print(f"\n  Reviews 5⭐ classificados como Negativo:")
casos = df[
    (df["rating"] == 5) &
    (df["classificacao_sentimento"] == "Negativo")
][["rating", "content", "sentimento_score"]].head(5)
display(casos)

print(f"\n  Reviews 1⭐ classificados como Positivo:")
casos2 = df[
    (df["rating"] == 1) &
    (df["classificacao_sentimento"] == "Positivo")
][["rating", "content", "sentimento_score"]].head(5)
display(casos2)

print(f"\n✅ Shape final: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\n📋 Todas as colunas:")
for col in df.columns:
    print(f"   - {col}")

RELATÓRIO DE VALIDAÇÃO — NLP

📊 SCORES
  Score médio geral:     0.7047
  Score reviews 1⭐:     0.0818
  Score reviews 5⭐:     0.7559

🏷️  CLASSIFICAÇÕES
  Positivo  : 148,755 (73.36%)
  Neutro    : 49,879 (24.60%)
  Negativo  : 4,151 (2.05%)

🔍 CASOS EXTREMOS

  Reviews 5⭐ classificados como Negativo:


,rating,content,sentimento_score
448,5,"ótimo para cabelo muito oleoso,refrescante!.",-1.0
749,5,o brasil não é pra amadores. está ficando tudo...,-1.0
811,5,"está vindo parece um gel,está diferente, estra...",-1.0
1201,5,tô usando há 15 dias e já notei diferença. dim...,-1.0
1279,5,estou usando há quase um ano consecutivo e a d...,-1.0



  Reviews 1⭐ classificados como Positivo:


,rating,content,sentimento_score
150,1,não gostei não.,1.0
421,1,não gostei uso essa marca mas não gostei este ...,1.0
444,1,o shampoo é ralo e o creme não cumpre o que pr...,1.0
510,1,"não superou as expectativas,não gostei.",1.0
864,1,não vale nada tudo engano dinheiro jogado fora.,1.0



✅ Shape final: 202,785 linhas × 14 colunas

📋 Todas as colunas:
   - date
   - rating
   - content
   - product_url
   - qtd_palavras
   - review_curto
   - content_original
   - produto_id
   - slug
   - nome_produto
   - categoria
   - tokens
   - sentimento_score
   - classificacao_sentimento


In [29]:
# ============================================================
# Salva o dataset final com NLP para o Notebook 05
# ============================================================

# Remove coluna auxiliar de tokens (lista — não serve pro Power BI)
df_export = df.drop(columns=["tokens"])

caminho_saida = PROCESSED_PATH / "reviews_nlp.csv"
df_export.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"✅ Dataset com NLP exportado com sucesso!")
print(f"📁 Caminho: {caminho_saida}")
print(f"📊 Shape final: {df_export.shape[0]:,} linhas × {df_export.shape[1]} colunas")
print(f"\n📋 Colunas exportadas:")
for col in df_export.columns:
    print(f"   - {col}")

✅ Dataset com NLP exportado com sucesso!
📁 Caminho: D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\processed\reviews_nlp.csv
📊 Shape final: 202,785 linhas × 13 colunas

📋 Colunas exportadas:
   - date
   - rating
   - content
   - product_url
   - qtd_palavras
   - review_curto
   - content_original
   - produto_id
   - slug
   - nome_produto
   - categoria
   - sentimento_score
   - classificacao_sentimento
